# 04 — Run MEFISTO

Fit MEFISTO once to the same full filtered cohort and save its sample factors, subject mean factors, and genus loadings.

MEFISTO models smooth variation along a continuous covariate such as age and supports grouped repeated measurements from multiple individuals.

**MEFISTO documentation:**  
https://biofam.github.io/MOFA2/MEFISTO.html

In [ ]:
from datetime import datetime
from pathlib import Path

import anndata as ad
import muon as mu
import numpy as np
import pandas as pd

# mefisto settings
N_FACTORS = 2
N_ITERATIONS = 1000
SEED = 2026

root = Path(".") if Path("data").exists() else Path("..")
source = sorted((root / "data" / "preprocessing").iterdir())[-1]
output = root / "data" / "mefisto" / datetime.now().strftime("%Y%m%d_%H%M%S")
output.mkdir(parents=True)

# load the filtered counts and aligned metadata
table = pd.read_csv(source / "counts_filtered.csv", dtype={"sample_id": str})
metadata = pd.read_csv(source / "metadata.csv", dtype={"sample_id": str, "subject_id": str})

counts = table.drop(columns="sample_id")

In [ ]:
# make rclr data with zeros left missing
values = counts.to_numpy(float)
rclr = np.full_like(values, np.nan)

for i, row in enumerate(values):
    positive = row > 0
    logged = np.log(row[positive])
    rclr[i, positive] = logged - logged.mean()

# describe the repeated longitudinal samples
obs = metadata.set_index("sample_id")[["subject_id", "age"]].copy()
obs["group"] = pd.factorize(obs["subject_id"])[0].astype(str)

# build the anndata object used by mefisto
adata = ad.AnnData(
    rclr,
    obs=obs,
    var=pd.DataFrame(index=counts.columns),
)

In [ ]:
# fit the mefisto model
mu.tl.mofa(
    adata,
    groups_label="group",
    likelihoods="gaussian",
    center_groups=False,
    n_factors=N_FACTORS,
    n_iterations=N_ITERATIONS,
    convergence_mode="medium",
    smooth_covariate="age",
    smooth_kwargs={
        "scale_cov": True,
        "sparseGP": False,
        "model_groups": False,
    },
    seed=SEED,
    outfile=str(output / "model.hdf5"),
)

In [ ]:
# format sample factor scores
factors = np.asarray(adata.obsm["X_mofa"])
names = [f"factor_{i+1}" for i in range(factors.shape[1])]

sample_factors = metadata[["sample_id", "subject_id", "age"]].copy()
sample_factors[names] = factors

# average sample factors within each subject
subject_scores = sample_factors.groupby("subject_id", as_index=False)[names].mean()

# format genus loadings
feature_loadings = pd.DataFrame(
    np.asarray(adata.varm["LFs"]),
    columns=names,
)
feature_loadings.insert(0, "feature_id", counts.columns)

In [ ]:
# save mefisto outputs
sample_factors.to_csv(output / "sample_factors.csv", index=False)
subject_scores.to_csv(output / "subject_scores.csv", index=False)
feature_loadings.to_csv(output / "feature_loadings.csv", index=False)

print("Saved", len(names), "factors to", output)